In [1]:
import torch
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchcrf import CRF
from torch.utils.data import DataLoader, Dataset
from transformers import BertModel, AutoTokenizer, AdamW
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

class BertNERCRF(nn.Module):
    def __init__(self, tokens_dim):
        super(BertNERCRF, self).__init__()
        
        if type(tokens_dim) != int:
            raise TypeError('tokens_dim should be an integer')
        if tokens_dim <= 0:
            raise ValueError('Classification layer dimension should be at least 1')

        self.tokens_dim = tokens_dim  # Number of labels
        self.bert = BertModel.from_pretrained("bert-base-cased")
        self.classifier = nn.Linear(self.bert.config.hidden_size, tokens_dim)
        self.crf = CRF(tokens_dim, batch_first=True)  # CRF layer

    def forward(self, input_ids, attention_mask, labels=None):
        """
        Forward pass with CRF decoding and loss computation.
        """
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state  # [batch_size, seq_len, hidden_dim]
        emissions = self.classifier(sequence_output)  # Convert hidden states to label scores

        if labels is not None:
            # Ensure the mask has the correct shape
            mask = (labels != 8).bool()  # Ignore padding tokens
            mask[:, 0] = True  # Ensure first timestep is always unmasked

            # Ensure labels are within valid range
            labels = torch.clamp(labels, min=0, max=self.tokens_dim - 1)

            loss = -self.crf(emissions, labels, mask=mask, reduction="mean")
            return loss
        else:
            # Decode predictions
            mask = attention_mask.bool()
            mask[:, 0] = True  # Ensure first timestep is always unmasked
            predictions = self.crf.decode(emissions, mask=mask)
            return predictions

    def save_pretrained(self, save_directory):
        """
        Save the pretrained model to the specified directory.
        """
        self.bert.save_pretrained(save_directory)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [5]:
import torch
from torch.utils.data import Dataset
import pandas as pd
import ast

class NerDataset(Dataset):
    """
    Custom dataset class for NER tasks with CRF.
    """

    def __init__(self, dataframe, tokenizer, label_map, max_len=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.label_map = label_map
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]
        text = str(row['Ingredient_Phrases'])
        labels_list = ast.literal_eval(row['Labels'])  # Convert string list to actual list

        encoding = self.tokenizer(
            text.split(),  # Tokenizing as split words
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
            is_split_into_words=True
        )

        word_ids = encoding.word_ids(batch_index=0)  # Maps tokens to words
        label_ids = [self.label_map['O']] * self.max_len  # Default to 'O' (not -1 or -100 for CRF)

        previous_word_id = None
        for i, word_id in enumerate(word_ids):
            if word_id is None:
                label_ids[i] = self.label_map['O']  # Assign 'O' label for padding or special tokens
            elif word_id != previous_word_id:
                label_ids[i] = self.label_map.get(labels_list[word_id], self.label_map['O'])  
            else:
                label_ids[i] = label_ids[i - 1]  # Ensure subwords have the same label
            previous_word_id = word_id

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label_ids, dtype=torch.long)
        }

# Updated Label Mapping for CRF
label_map = {
    'QUANTITY': 0,
    'UNIT': 1,
    'NAME': 2,
    'FORM': 3,
    'STATE': 4,
    'DF': 5,
    'SIZE': 6,
    'O': 7  # Assign 'O' a valid index (not -100 or -1)
}

# Padding labels should be mapped to 'O' (Index 7) instead of -1
PADDING_LABEL_ID = 8  # Set padding labels to 'O'
print(f"PADDING_LABEL_ID: {PADDING_LABEL_ID}")

from sklearn.model_selection import train_test_split
df = pd.read_csv('../../data/processed/10k_data_modified.csv')

PADDING_LABEL_ID: 8


- train + test split 

In [6]:
#ner_dataset = NerDataset(df, tokenizer, label_map)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = NerDataset(dataframe=train_df, tokenizer=tokenizer, label_map=label_map, max_len=128)
test_dataset = NerDataset(dataframe=test_df, tokenizer=tokenizer, label_map=label_map, max_len=128)

batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)


In [7]:
torch.cuda.set_device(1)  # Set GPU 1 as the active device
print("Now using GPU:", torch.cuda.current_device())

Now using GPU: 1


In [8]:
model = BertNERCRF(tokens_dim=len(label_map))
optimizer = AdamW(model.parameters(), lr=2e-5)

# Training settings
epochs = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.train()

/home/ritisha21089/miniconda3/lib/python3.9/site-packages/transformers/optimization.py:640: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


BertNERCRF(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_af

# MODEL TRAINING

In [9]:
from tqdm import tqdm

# Train
train_losses = []

for epoch in range(epochs):
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)

    for batch in progress_bar:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Ensure the first timestep is always 1 for CRF compatibility
        attention_mask[:, 0] = 1

        # Prepare mask for CRF
        mask = labels != 8  # Valid tokens (not padding)
        mask[:, 0] = True  # Ensure first timestep is unmasked

        # Forward pass
        loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

        # Backward pass
        loss.backward()
        optimizer.step()
        total_loss += loss.item()  # Accumulate loss

        # Update tqdm description with the latest loss
        progress_bar.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)  # Store loss

    print(f"Epoch {epoch+1}/{epochs}, Average Loss: {avg_loss:.4f}")

Epoch 1/5, Average Loss: 7.8022


Epoch 2/5, Average Loss: 1.5286


Epoch 3/5, Average Loss: 1.1682


Epoch 4/5, Average Loss: 0.9010


Epoch 5/5, Average Loss: 0.7297


# EVALUATION

In [13]:
model.eval()
predictions = []
true_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        predictions_batch = model(input_ids=input_ids, attention_mask=attention_mask)
        label_ids = labels.cpu().numpy()
        attention_mask_np = attention_mask.cpu().numpy()

        for i in range(len(label_ids)):
            true_label = []
            pred_label = []
            for j in range(len(label_ids[i])):
                if attention_mask_np[i][j] == 1 and label_ids[i][j] != -100:
                    true_label.append(label_ids[i][j])
                    pred_label.append(predictions_batch[i][j])
            true_labels.append(true_label)
            predictions.append(pred_label)

In [14]:
from sklearn.metrics import classification_report

flat_predictions_new = [item for sublist in predictions for item in sublist]
flat_true_labels_new = [item for sublist in true_labels for item in sublist]

precision = precision_score(flat_true_labels_new, flat_predictions_new, average='macro')
recall = recall_score(flat_true_labels_new, flat_predictions_new, average='macro')
f1 = f1_score(flat_true_labels_new, flat_predictions_new, average='macro')

print("Bert-base + CRF results:")
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

report = classification_report(flat_true_labels_new, flat_predictions_new, digits=4)
print(report)

Bert-base + CRF results:
Precision: 0.9524
Recall: 0.9682
F1-Score: 0.9601
              precision    recall  f1-score   support

           0     0.9877    0.9973    0.9925      3710
           1     0.9776    0.9974    0.9874      2672
           2     0.9654    0.9790    0.9722      6133
           3     0.9143    0.9052    0.9097      1002
           4     0.9579    0.9676    0.9627      3267
           5     0.9289    0.9839    0.9556       186
           6     0.8979    0.9410    0.9189       542
           7     0.9896    0.9743    0.9819     15245

    accuracy                         0.9764     32757
   macro avg     0.9524    0.9682    0.9601     32757
weighted avg     0.9766    0.9764    0.9764     32757

